In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from lightgbm import LGBMClassifier

In [2]:
# Load data
train_path = "data/train.csv"
test_path = "data/test.csv"
sample_sub_path = "data/sample_submission.csv"

In [3]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_sub_path)

In [4]:
target_col = "diagnosed_diabetes"
id_col = "id"

In [5]:
# Split features and target
X = train_df.drop(columns=[target_col]).copy()
y = train_df[target_col].copy()
X_test = test_df.copy()

In [6]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

num_cols = [col for col in num_cols if col in X.columns]
cat_cols = [col for col in cat_cols if col in X.columns]

X = X[num_cols + cat_cols].copy()
X_test = X_test[num_cols + cat_cols].copy()

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)
print("X shape:", X.shape)
print("X_test shape:", X_test.shape)

Numeric columns: ['id', 'age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history']
Categorical columns: ['gender', 'ethnicity', 'education_level', 'income_level', 'smoking_status', 'employment_status']
X shape: (700000, 25)
X_test shape: (300000, 25)


In [7]:
# Preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ],
    verbose_feature_names_out=False
)

In [8]:
# LightGBM pipeline
lgbm_model = LGBMClassifier(
    verbosity=-1,
    objective="binary",
    boosting_type="gbdt",
    random_state=42,
    n_jobs=-1
)

lgbm_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", lgbm_model)
])

# Hyperparameter search space
param_dist = {
    "model__num_leaves": [31, 50, 64, 80],
    "model__max_depth": [-1, 5, 6, 7],
    "model__learning_rate": [0.03, 0.05, 0.07],
    "model__n_estimators": [300, 500, 700],
    "model__min_child_samples": [20, 40, 60],
    "model__subsample": [0.7, 0.8, 0.9],
    "model__colsample_bytree": [0.7, 0.8, 0.9],
    "model__reg_alpha": [0.0, 0.5, 1.0],
    "model__reg_lambda": [0.0, 0.5, 1.0]
}

In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=lgbm_pipeline,
    param_distributions=param_dist,
    n_iter=12,
    scoring="roc_auc",
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

# Run search
random_search.fit(X, y)

print("Best CV score:", random_search.best_score_)
print("Best params:")

for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")

Fitting 5 folds for each of 12 candidates, totalling 60 fits


/Users/heosunghak/miniforge3/envs/titanic/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/heosunghak/miniforge3/envs/titanic/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/heosunghak/miniforge3/envs/titanic/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/heosunghak/miniforge3/envs/titanic/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/heosunghak/miniforge3/envs/titanic/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning

Best CV score: 0.7256695733960667
Best params:
  model__subsample: 0.9
  model__reg_lambda: 0.5
  model__reg_alpha: 1.0
  model__num_leaves: 80
  model__n_estimators: 500
  model__min_child_samples: 60
  model__max_depth: -1
  model__learning_rate: 0.03
  model__colsample_bytree: 0.7


In [11]:
# Best model
best_pipeline = random_search.best_estimator_

# Predict test
test_pred_proba = best_pipeline.predict_proba(X_test)[:, 1]

# Save submission
submission = sample_sub.copy()
submission.iloc[:, 1] = submission.iloc[:, 1].astype(float)
submission.iloc[:, 1] = test_pred_proba
submission.to_csv("submissions/lightgbm_with_id_random_search.csv", index=False)

print("Saved: submissions/lightgbm_with_id_random_search.csv")

/Users/heosunghak/miniforge3/envs/titanic/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/var/folders/m_/gq3fgb6x2rs_xpbs_2llxbvc0000gn/T/ipykernel_46180/3616782647.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0         0.0
1         0.0
2         0.0
3         0.0
4         0.0
         ... 
299995    0.0
299996    0.0
299997    0.0
299998    0.0
299999    0.0
Name: diagnosed_diabetes, Length: 300000, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  submission.iloc[:, 1] = submission.iloc[:, 1].astype(float)


Saved: submissions/lightgbm_with_id_random_search.csv


In [12]:
# Feature importance
feature_names = best_pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = best_pipeline.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print("\nTop 30 feature importances:")
print(importance_df.head(30))


Top 30 feature importances:
                               feature  importance
3   physical_activity_minutes_per_week        9192
15                       triglycerides        4038
1                                  age        2633
7                                  bmi        2554
12                   cholesterol_total        2161
14                     ldl_cholesterol        2155
9                          systolic_bp        1933
4                           diet_score        1895
0                                   id        1884
6            screen_time_hours_per_day        1684
11                          heart_rate        1671
13                     hdl_cholesterol        1566
8                   waist_to_hip_ratio        1261
5                  sleep_hours_per_day        1200
10                        diastolic_bp        1192
2         alcohol_consumption_per_week         362
16             family_history_diabetes         276
27            education_level_Graduate         124
18